# Modern: Amazon Nova 2 Sonic Speech-to-Speech

Streams an audio file to **Amazon Nova 2 Sonic** via Bedrock's bidirectional streaming API and uploads Nova's spoken response to S3.

```
input audio  →  Nova 2 Sonic (Bedrock)  →  nova-audio/output.mp3  (S3)
```

> Running inside SageMaker — credentials come from the instance role automatically.

**Required IAM permissions on your SageMaker execution role:**
- `bedrock:InvokeModelWithBidirectionalStream` on `amazon.nova-2-sonic-v1:0`
- `s3:PutObject` on your target bucket

**Supported regions:** `us-east-1`, `us-west-2`, `eu-north-1`, `ap-northeast-1`

## 1. Install Dependencies

In [ ]:
%pip install -q aws-sdk-bedrock-runtime smithy-aws-core

In [ ]:
# Install ffmpeg if not already present
import subprocess, shutil
if not shutil.which("ffmpeg"):
    subprocess.run(["sudo", "apt-get", "install", "-y", "-q", "ffmpeg"], check=True)
    print("ffmpeg installed")
else:
    print("ffmpeg already available")

## 2. Configuration

In [ ]:
# ── Set these before running ─────────────────────────────────────────────────
S3_BUCKET_NAME   = "your-bucket-name"        # <-- replace
OUTPUT_KEY       = "nova-audio/output.mp3"
AWS_REGION       = "us-east-1"               # must be a Nova 2 Sonic supported region
INPUT_AUDIO_FILE = "sample_input.wav"        # upload this file to the same directory
# ─────────────────────────────────────────────────────────────────────────────

MODEL_ID          = "amazon.nova-2-sonic-v1:0"
INPUT_SAMPLE_RATE = 16000
OUTPUT_SAMPLE_RATE = 24000
SAMPLE_SIZE_BITS  = 16
CHANNELS          = 1
CHUNK_SIZE        = 1024 * 4

## 3. Helper Functions

In [ ]:
import struct, subprocess, tempfile

def convert_to_pcm_wav(input_path: str) -> str:
    """Convert any audio file to 16kHz mono 16-bit PCM WAV using ffmpeg."""
    out = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    out.close()
    result = subprocess.run(
        ["ffmpeg", "-y", "-i", input_path, "-ar", str(INPUT_SAMPLE_RATE),
         "-ac", "1", "-sample_fmt", "s16", out.name],
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{result.stderr.decode()}")
    return out.name

def read_pcm_from_wav(wav_path: str) -> bytes:
    """Strip the 44-byte WAV header and return raw PCM bytes."""
    with open(wav_path, "rb") as f:
        return f.read()[44:]

def pcm_to_wav_bytes(pcm_bytes: bytes, sample_rate: int) -> bytes:
    byte_rate   = sample_rate * CHANNELS * (SAMPLE_SIZE_BITS // 8)
    block_align = CHANNELS * (SAMPLE_SIZE_BITS // 8)
    data_size   = len(pcm_bytes)
    header = struct.pack(
        "<4sI4s4sIHHIIHH4sI",
        b"RIFF", 36 + data_size, b"WAVE",
        b"fmt ", 16, 1, CHANNELS,
        sample_rate, byte_rate, block_align, SAMPLE_SIZE_BITS,
        b"data", data_size,
    )
    return header + pcm_bytes

def pcm_to_mp3(pcm_bytes: bytes) -> bytes:
    """Convert raw PCM output from Nova 2 Sonic to MP3 via ffmpeg."""
    wav_bytes = pcm_to_wav_bytes(pcm_bytes, OUTPUT_SAMPLE_RATE)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as wf:
        wf.write(wav_bytes)
        wav_path = wf.name
    mp3_path = wav_path.replace(".wav", ".mp3")
    result = subprocess.run(
        ["ffmpeg", "-y", "-i", wav_path, "-codec:a", "libmp3lame", "-q:a", "2", mp3_path],
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg MP3 conversion failed:\n{result.stderr.decode()}")
    with open(mp3_path, "rb") as f:
        return f.read()

## 4. Speech-to-Speech via Nova 2 Sonic

In [ ]:
import asyncio, base64, json, uuid
from aws_sdk_bedrock_runtime.client import (
    BedrockRuntimeClient,
    InvokeModelWithBidirectionalStreamOperationInput,
)
from aws_sdk_bedrock_runtime.config import Config, HTTPAuthSchemeResolver, SigV4AuthScheme
from aws_sdk_bedrock_runtime.models import (
    BidirectionalInputPayloadPart,
    InvokeModelWithBidirectionalStreamInputChunk,
)
from smithy_aws_core.identity import EnvironmentCredentialsResolver

async def speech_to_speech(pcm_input: bytes) -> bytes:
    config = Config(
        endpoint_uri=f"https://bedrock-runtime.{AWS_REGION}.amazonaws.com",
        region=AWS_REGION,
        aws_credentials_identity_resolver=EnvironmentCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="bedrock")},
    )
    client = BedrockRuntimeClient(config=config)

    prompt_name       = str(uuid.uuid4())
    audio_content_name  = str(uuid.uuid4())
    system_content_name = str(uuid.uuid4())

    stream = await client.invoke_model_with_bidirectional_stream(
        InvokeModelWithBidirectionalStreamOperationInput(model_id=MODEL_ID)
    )

    async def send(payload: dict):
        chunk = InvokeModelWithBidirectionalStreamInputChunk(
            value=BidirectionalInputPayloadPart(bytes_=json.dumps(payload).encode("utf-8"))
        )
        await stream.input_stream.send(chunk)

    await send({"event": {"sessionStart": {"inferenceConfiguration": {"maxTokens": 1024, "topP": 0.9, "temperature": 0.7}, "turnDetectionConfiguration": {"endpointingSensitivity": "HIGH"}}}})
    await send({"event": {"promptStart": {"promptName": prompt_name, "textOutputConfiguration": {"mediaType": "text/plain"}, "audioOutputConfiguration": {"mediaType": "audio/lpcm", "sampleRateHertz": OUTPUT_SAMPLE_RATE, "sampleSizeBits": SAMPLE_SIZE_BITS, "channelCount": CHANNELS, "voiceId": "matthew", "encoding": "base64", "audioType": "SPEECH"}}}})

    await send({"event": {"contentStart": {"promptName": prompt_name, "contentName": system_content_name, "type": "TEXT", "interactive": False, "role": "SYSTEM", "textInputConfiguration": {"mediaType": "text/plain"}}}})
    await send({"event": {"textInput": {"promptName": prompt_name, "contentName": system_content_name, "content": "You are a friendly course narrator for Pixel Learning Co. Listen to the user's audio and respond with a clear, engaging spoken summary suitable for an online course. Keep it concise."}}})
    await send({"event": {"contentEnd": {"promptName": prompt_name, "contentName": system_content_name}}})

    await send({"event": {"contentStart": {"promptName": prompt_name, "contentName": audio_content_name, "type": "AUDIO", "interactive": True, "role": "USER", "audioInputConfiguration": {"mediaType": "audio/lpcm", "sampleRateHertz": INPUT_SAMPLE_RATE, "sampleSizeBits": SAMPLE_SIZE_BITS, "channelCount": CHANNELS, "audioType": "SPEECH", "encoding": "base64"}}}})

    print(f"Streaming {len(pcm_input):,} bytes in {CHUNK_SIZE}-byte chunks...")
    for i in range(0, len(pcm_input), CHUNK_SIZE):
        encoded = base64.b64encode(pcm_input[i:i + CHUNK_SIZE]).decode("utf-8")
        await send({"event": {"audioInput": {"promptName": prompt_name, "contentName": audio_content_name, "content": encoded}}})

    await send({"event": {"contentEnd": {"promptName": prompt_name, "contentName": audio_content_name}}})
    await send({"event": {"promptEnd": {"promptName": prompt_name}}})
    await send({"event": {"sessionEnd": {}}})
    await stream.input_stream.close()

    audio_chunks = []
    try:
        while True:
            output = await stream.await_output()
            result = await output[1].receive()
            if result.value and result.value.bytes_:
                data  = json.loads(result.value.bytes_.decode("utf-8"))
                event = data.get("event", {})
                if "audioOutput" in event:
                    audio_chunks.append(base64.b64decode(event["audioOutput"]["content"]))
                elif "textOutput" in event:
                    print(f"Nova said: {event['textOutput'].get('content', '')}")
    except Exception:
        pass

    return b"".join(audio_chunks)

## 5. Run the Pipeline

In [ ]:
import boto3

print("Converting input audio to 16kHz mono PCM WAV...")
converted_wav = convert_to_pcm_wav(INPUT_AUDIO_FILE)
pcm_input     = read_pcm_from_wav(converted_wav)

print("Starting speech-to-speech via Nova 2 Sonic...")
pcm_output = asyncio.run(speech_to_speech(pcm_input))

if not pcm_output:
    raise RuntimeError("No audio received. Check your input audio and IAM permissions.")

print("Converting response PCM → MP3...")
mp3_bytes = pcm_to_mp3(pcm_output)

print(f"Uploading to s3://{S3_BUCKET_NAME}/{OUTPUT_KEY} ...")
s3 = boto3.client("s3", region_name=AWS_REGION)
s3.put_object(Bucket=S3_BUCKET_NAME, Key=OUTPUT_KEY, Body=mp3_bytes, ContentType="audio/mpeg")
print(f"Done. Audio uploaded to s3://{S3_BUCKET_NAME}/{OUTPUT_KEY}")

## 6. Preview the Response Audio (optional)

In [ ]:
import IPython.display as ipd
ipd.Audio(mp3_bytes)